# ConversationAgent Testing Notebook

This notebook demonstrates and tests the DSPy ConversationAgent module.

## Important Notes
- **If you encounter AttributeError**: Restart the kernel (Kernel → Restart) and run all cells from the beginning
- The notebook automatically reloads modules to pick up the latest code changes
- Make sure you have a `.env` file with the required configuration (copy from `.env.example`)

## Setup
Required environment variables in `.env`:
- `GEMINI_API_KEY` (required)
- `DEFAULT_LLM_MODEL` (optional, defaults to gemini/gemini-2.5-flash)
- `LLM_TEMPERATURE` (optional, defaults to 0.7)
- `LLM_MAX_TOKENS` (optional, defaults to 1024)
- `SECRET_KEY` (required, min 32 characters)
- `DATABASE_URL` (required)
- Other settings as shown in `.env.example`

In [1]:
# Import required modules
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Load environment variables
from dotenv import load_dotenv
load_dotenv(override=True)  # Force reload of .env

# Force reload modules to pick up latest changes
import importlib
if 'src.infrastructure.llm.conversation_agent' in sys.modules:
    importlib.reload(sys.modules['src.infrastructure.llm.conversation_agent'])
if 'src.application.interfaces.conversation_agent' in sys.modules:
    importlib.reload(sys.modules['src.application.interfaces.conversation_agent'])
if 'src.config.settings' in sys.modules:
    importlib.reload(sys.modules['src.config.settings'])

print("✅ Environment loaded and modules reloaded")

✅ Environment loaded and modules reloaded


In [2]:
# Import ConversationAgent components
try:
    from src.infrastructure.llm.conversation_agent import ConversationAgent
    from src.application.interfaces.conversation_agent import (
        ConversationContext,
        Message,
        AgentResponse,
    )
    print("✅ Imports successful")
except Exception as e:
    print(f"❌ Import failed: {e}")
    print("\nTroubleshooting:")
    print("1. Make sure you have a .env file with all required settings")
    print("2. Restart the Jupyter kernel (Kernel → Restart)")
    print("3. Run cells in order from the beginning")
    raise

✅ Imports successful


## 1. Basic Initialization

Create a ConversationAgent instance with default configuration.

In [3]:
# Initialize the agent
try:
    agent = ConversationAgent()
    print(f"Agent initialized with model: {agent.model}")
    print(f"Temperature: {agent.temperature}")
    print(f"Max tokens: {agent.max_tokens}")
except AttributeError as e:
    print(f"❌ AttributeError: {e}")
    print("\n⚠️  This usually means the Jupyter kernel has cached old code.")
    print("Solution: Restart the kernel (Kernel → Restart & Run All)")
    raise
except Exception as e:
    print(f"❌ Initialization failed: {e}")
    print("\nCheck your .env file has all required settings")
    raise

Agent initialized with model: gemini/gemini-2.5-flash
Temperature: 0.7
Max tokens: 1024


## 2. Simple Conversation (No History)

Test a basic single-turn conversation.

In [4]:
# Create a simple context
context = ConversationContext(
    current_message="Hello! Can you tell me a fun fact about Python programming?"
)

# Generate response
response = agent.generate_response(context)

print("=" * 80)
print("USER MESSAGE:")
print(context.current_message)
print("\n" + "=" * 80)
print("AGENT RESPONSE:")
print(response.content)
print("\n" + "=" * 80)
print("REASONING:")
print(response.reasoning)
print("=" * 80)

USER MESSAGE:
Hello! Can you tell me a fun fact about Python programming?

AGENT RESPONSE:
Hello! Here's a fun fact for you: Python was named after the British comedy group Monty Python, not the snake! Guido van Rossum, Python's creator, was a big fan of "Monty Python's Flying Circus."

REASONING:
The user asked for a fun fact about Python programming. The trajectory indicates that a direct response was chosen. I will provide a fun and interesting fact about Python.


## 3. Multi-Turn Conversation (With History)

Test conversation with context from previous messages.

In [5]:
# Create conversation history
history = (
    Message(role="user", content="What is the capital of France?"),
    Message(role="assistant", content="The capital of France is Paris. It's known for the Eiffel Tower, the Louvre Museum, and its rich cultural heritage."),
    Message(role="user", content="What's the population?"),
    Message(role="assistant", content="Paris has a population of approximately 2.2 million people within the city limits, and about 12 million in the metropolitan area."),
)

# Create context with history
context_with_history = ConversationContext(
    current_message="What famous landmarks can I visit there?",
    history=history
)

# Generate response
response = agent.generate_response(context_with_history)

print("=" * 80)
print("CONVERSATION HISTORY:")
for msg in history:
    print(f"\n[{msg.role.upper()}]: {msg.content}")
print("\n" + "=" * 80)
print("NEW USER MESSAGE:")
print(context_with_history.current_message)
print("\n" + "=" * 80)
print("AGENT RESPONSE:")
print(response.content)
print("=" * 80)

CONVERSATION HISTORY:

[USER]: What is the capital of France?

[ASSISTANT]: The capital of France is Paris. It's known for the Eiffel Tower, the Louvre Museum, and its rich cultural heritage.

[USER]: What's the population?

[ASSISTANT]: Paris has a population of approximately 2.2 million people within the city limits, and about 12 million in the metropolitan area.

NEW USER MESSAGE:
What famous landmarks can I visit there?

AGENT RESPONSE:
In Paris, you can visit many famous landmarks! Of course, there's the iconic Eiffel Tower and the world-renowned Louvre Museum. Other must-see sites include the Notre-Dame Cathedral, the Arc de Triomphe, the Sacré-Cœur Basilica, and the Palace of Versailles which is just outside the city.


## 4. WhatsApp-Style Conversation Simulation

Simulate a realistic WhatsApp conversation flow.

In [6]:
# Simulate a WhatsApp conversation
conversation_history = []

def chat(user_message: str) -> str:
    """Helper function to simulate chat interaction."""
    # Create context from conversation history
    context = ConversationContext(
        current_message=user_message,
        history=tuple(conversation_history) if conversation_history else None
    )
    
    # Get response
    response = agent.generate_response(context)
    
    # Update history
    conversation_history.append(Message(role="user", content=user_message))
    conversation_history.append(Message(role="assistant", content=response.content))
    
    return response.content

# Start conversation
print("🤖 WhatsApp Chatbot Simulation\n")
print("=" * 80)

🤖 WhatsApp Chatbot Simulation



In [7]:
# Turn 1
user_msg = "Hi! I need help planning a trip to Japan."
print(f"👤 USER: {user_msg}")
bot_response = chat(user_msg)
print(f"🤖 BOT: {bot_response}")
print("\n" + "=" * 80)

👤 USER: Hi! I need help planning a trip to Japan.
🤖 BOT: Hi there! I'd be happy to help you plan your trip to Japan. To get started, could you tell me a bit more about what you're looking for? For example:

*   **When are you planning to go?** (e.g., specific dates, season)
*   **How long will your trip be?**
*   **What's your estimated budget?** (e.g., budget, mid-range, luxury)
*   **What are your main interests?** (e.g., culture, food, nature, cities, shopping, specific attractions)
*   **Who are you traveling with?** (e.g., solo, family with kids, couple, friends)

The more details you can provide, the better I can assist you!



In [8]:
# Turn 2
user_msg = "I'm interested in visiting during cherry blossom season. When is that?"
print(f"👤 USER: {user_msg}")
bot_response = chat(user_msg)
print(f"🤖 BOT: {bot_response}")
print("\n" + "=" * 80)

👤 USER: I'm interested in visiting during cherry blossom season. When is that?
🤖 BOT: Cherry blossom season in Japan typically runs from late March to early April, though this can vary by region. For example, they bloom earlier in the warmer southern regions (like Kyushu) and later in the colder northern regions (like Hokkaido) and at higher elevations. The exact timing also shifts slightly year by year depending on the weather.

To help me narrow down recommendations, do you have a specific part of Japan you're most interested in visiting?



In [9]:
# Turn 3
user_msg = "What cities should I visit?"
print(f"👤 USER: {user_msg}")
bot_response = chat(user_msg)
print(f"🤖 BOT: {bot_response}")
print("\n" + "=" * 80)

👤 USER: What cities should I visit?
🤖 BOT: That's a great question! Japan has many beautiful cities that offer incredible cherry blossom experiences. To help me recommend the best ones for you, could you tell me a bit about what you're hoping to experience in a city during cherry blossom season? For example:

*   Are you looking for vibrant city life with famous landmarks and bustling parks?
*   Do you prefer a more traditional atmosphere with temples, gardens, and historical sites?
*   Are you interested in a specific type of experience, like nighttime illuminations, boat rides under the blossoms, or combining it with other activities like food or shopping?
*   Do you prefer larger cities or smaller, more quaint towns?

Once I have a better idea of your interests, I can suggest some perfect spots!



In [10]:
# View full conversation history
print("📜 FULL CONVERSATION HISTORY:\n")
for i, msg in enumerate(conversation_history, 1):
    icon = "👤" if msg.role == "user" else "🤖"
    print(f"{i}. {icon} [{msg.role.upper()}]: {msg.content}\n")

📜 FULL CONVERSATION HISTORY:

1. 👤 [USER]: Hi! I need help planning a trip to Japan.

2. 🤖 [ASSISTANT]: Hi there! I'd be happy to help you plan your trip to Japan. To get started, could you tell me a bit more about what you're looking for? For example:

*   **When are you planning to go?** (e.g., specific dates, season)
*   **How long will your trip be?**
*   **What's your estimated budget?** (e.g., budget, mid-range, luxury)
*   **What are your main interests?** (e.g., culture, food, nature, cities, shopping, specific attractions)
*   **Who are you traveling with?** (e.g., solo, family with kids, couple, friends)

The more details you can provide, the better I can assist you!

3. 👤 [USER]: I'm interested in visiting during cherry blossom season. When is that?

4. 🤖 [ASSISTANT]: Cherry blossom season in Japan typically runs from late March to early April, though this can vary by region. For example, they bloom earlier in the warmer southern regions (like Kyushu) and later in the colder

## 5. Custom Configuration

Test agent with custom configuration.

In [11]:
# Create agent with custom config for more creative responses
creative_agent = ConversationAgent(
    model="gemini/gemini-2.5-flash",
    temperature=0.9,  # Higher temperature for more creativity
    max_tokens=2048,
)

print(f"Creative Agent Config:")
print(f"- Model: {creative_agent.model}")
print(f"- Temperature: {creative_agent.temperature}")
print(f"- Max Tokens: {creative_agent.max_tokens}")

Creative Agent Config:
- Model: gemini/gemini-2.5-flash
- Temperature: 0.9
- Max Tokens: 2048


In [12]:
# Test creative agent
creative_context = ConversationContext(
    current_message="Write a short poem about coding in Python."
)

creative_response = creative_agent.generate_response(creative_context)

print("=" * 80)
print("CREATIVE RESPONSE:")
print(creative_response.content)
print("=" * 80)

CREATIVE RESPONSE:
In Python's realm, where logic flows,
With simple lines, a program grows.
No curly braces, just a neat indent,
A coder's joy, a language heaven-sent.

From web to AI, a versatile grace,
Solving problems with a smiling face.
"Import this," the zen will say,
Guiding thoughts throughout the day.


## 6. Error Handling Tests

Test validation and error scenarios.

In [13]:
# Test empty message validation
try:
    invalid_context = ConversationContext(current_message="")
    print("❌ Should have raised ValueError")
except ValueError as e:
    print(f"✅ Correctly caught empty message: {e}")

✅ Correctly caught empty message: Current message cannot be empty


In [14]:
# Test whitespace-only message
try:
    invalid_context = ConversationContext(current_message="   \n\t  ")
    print("❌ Should have raised ValueError")
except ValueError as e:
    print(f"✅ Correctly caught whitespace-only message: {e}")

❌ Should have raised ValueError


In [15]:
# Test invalid message role
try:
    invalid_message = Message(role="invalid", content="test")
    print("❌ Should have raised ValueError")
except ValueError as e:
    print(f"✅ Correctly caught invalid role: {e}")

✅ Correctly caught invalid role: Invalid role: invalid. Must be 'user', 'assistant', or 'system'


## 7. Performance Testing

Measure response times.

In [16]:
import time

# Test response time
test_messages = [
    "What is 2+2?",
    "Tell me a joke.",
    "What's the weather like in Tokyo?",
]

times = []

for msg in test_messages:
    context = ConversationContext(current_message=msg)
    
    start = time.time()
    response = agent.generate_response(context)
    elapsed = time.time() - start
    
    times.append(elapsed)
    print(f"⏱️  '{msg[:30]}...' - {elapsed:.2f}s")

print(f"\n📊 Average response time: {sum(times)/len(times):.2f}s")

⏱️  'What is 2+2?...' - 14.24s
⏱️  'Tell me a joke....' - 4.75s
⏱️  'What's the weather like in Tok...' - 4.26s

📊 Average response time: 7.75s


## 8. Runtime Reconfiguration

Test changing configuration at runtime.

In [17]:
# Create agent with default config
reconfigurable_agent = ConversationAgent()

print(f"Initial temperature: {reconfigurable_agent.temperature}")

# Test with initial config
context = ConversationContext(current_message="Tell me about AI.")
response1 = reconfigurable_agent.generate_response(context)
print(f"\nResponse 1 (temp={reconfigurable_agent.temperature}):")
print(response1.content[:100] + "...")

Initial temperature: 0.7

Response 1 (temp=0.7):
Artificial Intelligence (AI) is a broad field of computer science that aims to create machines capab...


In [18]:
# Reconfigure with different temperature
reconfigurable_agent.configure(
    model="gemini/gemini-2.5-flash",
    temperature=0.1,  # More deterministic
    max_tokens=1024,
)

print(f"New temperature: {reconfigurable_agent.temperature}")

# Test with new config
response2 = reconfigurable_agent.generate_response(context)
print(f"\nResponse 2 (temp={reconfigurable_agent.temperature}):")
print(response2.content[:100] + "...")

New temperature: 0.1

Response 2 (temp=0.1):
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are pro...


## Summary

This notebook demonstrated:
- ✅ Basic agent initialization
- ✅ Single-turn conversations
- ✅ Multi-turn conversations with history
- ✅ WhatsApp-style conversation simulation
- ✅ Custom configuration
- ✅ Error handling and validation
- ✅ Performance measurement
- ✅ Runtime reconfiguration

The ConversationAgent is ready for production use! 🚀